In [65]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import sys
print(sys.version)
print(sys.executable)

3.11.15 (main, Mar 11 2026, 17:14:47) [Clang 20.1.8 ]
/opt/anaconda3/envs/hawkes/bin/python


In [66]:
btc = pd.read_csv('data/btcusdt_mo.csv')
eth = pd.read_csv('data/ethusdt_mo.csv')
print(btc.shape, eth.shape)
print(btc['event_type'].value_counts())
print(btc.head())

(4328919, 5) (3501949, 5)
event_type
0    2236511
1    2092408
Name: count, dtype: int64
       timestamp_us  event_type     price  quantity   symbol
0  1778445142858897           0  81088.84   0.00435  BTCUSDT
1  1778445142920893           1  81088.83   0.10236  BTCUSDT
2  1778445142924136           1  81088.83   0.00085  BTCUSDT
3  1778445142924137           1  81088.82   0.00070  BTCUSDT
4  1778445142924138           1  81088.72   0.00014  BTCUSDT


In [67]:
# Convert microseconds to seconds
btc['t'] = btc['timestamp_us'] / 1e6
eth['t'] = eth['timestamp_us'] / 1e6

btc_buy  = btc[btc['event_type'] == 0]['t'].values
btc_sell = btc[btc['event_type'] == 1]['t'].values
eth_buy  = eth[eth['event_type'] == 0]['t'].values
eth_sell = eth[eth['event_type'] == 1]['t'].values

print(f"BTC buy: {len(btc_buy):,}  sell: {len(btc_sell):,}")
print(f"ETH buy: {len(eth_buy):,}  sell: {len(eth_sell):,}")
print(f"Time span: {(btc['t'].max() - btc['t'].min())/3600:.1f} hours")

BTC buy: 2,236,511  sell: 2,092,408
ETH buy: 1,692,944  sell: 1,809,005
Time span: 152.9 hours


In [68]:
# Take first hour only to test
t0 = btc['t'].min()
mask_buy  = (btc_buy  >= t0) & (btc_buy  < t0 + 3600)
mask_sell = (btc_sell >= t0) & (btc_sell < t0 + 3600)

T_B = btc_buy[mask_buy]   - t0
T_S = btc_sell[mask_sell] - t0

print(f"Events in first hour — buy: {len(T_B)}, sell: {len(T_S)}")

Events in first hour — buy: 30385, sell: 31626


In [69]:
from numba import njit
import numpy as np
from scipy.optimize import minimize

@njit
def soe_loglik_fast(params, times, types, T_B, T_S, beta1, beta2):
    mu_B  = np.exp(min(max(params[0], -15), 15))
    mu_S  = np.exp(min(max(params[1], -15), 15))
    a_BB1 = np.exp(min(max(params[2], -15), 15))
    a_SS1 = np.exp(min(max(params[3], -15), 15))
    a_BS1 = np.exp(min(max(params[4], -15), 15))
    a_SB1 = np.exp(min(max(params[5], -15), 15))
    a_BB2 = np.exp(min(max(params[6], -15), 15))
    a_SS2 = np.exp(min(max(params[7], -15), 15))
    a_BS2 = np.exp(min(max(params[8], -15), 15))
    a_SB2 = np.exp(min(max(params[9], -15), 15))

    A_B1 = A_S1 = A_B2 = A_S2 = 0.0
    last_t = 0.0
    loglik = 0.0

    for i in range(len(times)):
        t     = times[i]
        etype = types[i]
        dt    = t - last_t
        if dt > 0.0:
            d1 = np.exp(-beta1 * dt)
            d2 = np.exp(-beta2 * dt)
            A_B1 *= d1;  A_S1 *= d1
            A_B2 *= d2;  A_S2 *= d2

        if etype == 0.0:
            lam = mu_B + a_BB1*A_B1 + a_SB1*A_S1 + a_BB2*A_B2 + a_SB2*A_S2
            if lam <= 1e-300:
                return 1e10
            loglik += np.log(lam)
            A_B1 += 1.0;  A_B2 += 1.0
        else:
            lam = mu_S + a_BS1*A_B1 + a_SS1*A_S1 + a_BS2*A_B2 + a_SS2*A_S2
            if lam <= 1e-300:
                return 1e10
            loglik += np.log(lam)
            A_S1 += 1.0;  A_S2 += 1.0
        last_t = t

    T_end = times[-1]
    for i in range(len(T_B)):
        v1 = 1.0 - np.exp(-beta1*(T_end - T_B[i]))
        v2 = 1.0 - np.exp(-beta2*(T_end - T_B[i]))
        loglik -= (a_BB1/beta1)*v1 + (a_BB2/beta2)*v2
        loglik -= (a_BS1/beta1)*v1 + (a_BS2/beta2)*v2
    for i in range(len(T_S)):
        v1 = 1.0 - np.exp(-beta1*(T_end - T_S[i]))
        v2 = 1.0 - np.exp(-beta2*(T_end - T_S[i]))
        loglik -= (a_SS1/beta1)*v1 + (a_SS2/beta2)*v2
        loglik -= (a_SB1/beta1)*v1 + (a_SB2/beta2)*v2
    loglik -= (mu_B + mu_S) * T_end

    return -loglik


def fit_window_soe_fast(T_B, T_S, beta1=100.0, beta2=1.0, n_inits=3):
    if len(T_B) < 20 or len(T_S) < 20:
        return None

    times = np.concatenate([T_B, T_S])
    types = np.concatenate([np.zeros(len(T_B)), np.ones(len(T_S))])
    order = np.argsort(times, kind='stable')
    times = times[order].astype(np.float64)
    types = types[order].astype(np.float64)
    T_B   = np.sort(T_B).astype(np.float64)
    T_S   = np.sort(T_S).astype(np.float64)

    def obj(params):
        return soe_loglik_fast(params, times, types, T_B, T_S, beta1, beta2)

    rate_B  = len(T_B) / (T_B[-1] - T_B[0] + 1e-6)
    rate_S  = len(T_S) / (T_S[-1] - T_S[0] + 1e-6)
    x0_base = np.array([np.log(rate_B*0.1), np.log(rate_S*0.1),
                         np.log(0.2), np.log(0.2), np.log(0.05), np.log(0.05),
                         np.log(0.2), np.log(0.2), np.log(0.05), np.log(0.05)])

    best_val, best_x = np.inf, None
    for i in range(n_inits):
        noise = np.random.randn(10)*0.5 if i > 0 else np.zeros(10)
        x0 = x0_base + noise
        try:
            res = minimize(obj, x0, method='L-BFGS-B',
                           options={'maxiter': 300})
            if np.isfinite(res.fun) and res.fun < best_val:
                best_val, best_x = res.fun, res.x
        except:
            continue

    if best_x is None:
        return None

    p = np.clip(best_x, -15, 15)
    a_BB1,a_SS1,a_BS1,a_SB1 = np.exp(p[2]),np.exp(p[3]),np.exp(p[4]),np.exp(p[5])
    a_BB2,a_SS2,a_BS2,a_SB2 = np.exp(p[6]),np.exp(p[7]),np.exp(p[8]),np.exp(p[9])
    phi_mat = (np.array([[a_BB1,a_SB1],[a_BS1,a_SS1]])/beta1 +
               np.array([[a_BB2,a_SB2],[a_BS2,a_SS2]])/beta2)
    eta = float(np.max(np.abs(np.linalg.eigvals(phi_mat))))

    return {
        'mu_B': float(np.exp(p[0])), 'mu_S': float(np.exp(p[1])), #bug fix: return mue so that GOF doesn't use default values. 
        'phi_BB1': a_BB1/beta1, 'phi_SS1': a_SS1/beta1,
        'phi_BS1': a_BS1/beta1, 'phi_SB1': a_SB1/beta1,
        'phi_BB2': a_BB2/beta2, 'phi_SS2': a_SS2/beta2,
        'phi_BS2': a_BS2/beta2, 'phi_SB2': a_SB2/beta2,
        'eta': eta, 'loglik': -best_val
    }


# install numba if needed
# pip install numba

# warm up numba JIT (first call compiles, takes ~30s)
print("Warming up numba...")
_ = fit_window_soe_fast(T_B[:500], T_S[:500])
print("Done. Now timing full window...")

import time
t0 = time.time()
result = fit_window_soe_fast(T_B, T_S)
print(f"Time: {time.time()-t0:.1f}s")
print(result)

Warming up numba...
Done. Now timing full window...
Time: 2.4s
{'mu_B': 1.5121388608439645, 'mu_S': 0.8531804522849796, 'phi_BB1': np.float64(0.7519491824471978), 'phi_SS1': np.float64(0.8422403020150152), 'phi_BS1': np.float64(0.0004636388929520609), 'phi_SB1': np.float64(0.00060983319921226), 'phi_BB2': np.float64(0.03622311076622021), 'phi_SS2': np.float64(0.046271098630149055), 'phi_BS2': np.float64(0.01856581568598074), 'phi_SB2': np.float64(0.027262642779756575), 'eta': 0.8935449458975018, 'loglik': 270380.0281127154}


In [70]:

@njit
def simulate_bivariate_soe(mu_B, mu_S, 
                            a_BB1, a_SS1, a_BS1, a_SB1,
                            a_BB2, a_SS2, a_BS2, a_SB2,
                            beta1, beta2, T_max):
    """Ogata thinning for bivariate SoE Hawkes"""
    T_B, T_S = [], []
    A_B1 = A_S1 = A_B2 = A_S2 = 0.0
    t = 0.0

    while t < T_max:
        # upper bound on total intensity
        lam_B = mu_B + a_BB1*A_B1 + a_SB1*A_S1 + a_BB2*A_B2 + a_SB2*A_S2
        lam_S = mu_S + a_BS1*A_B1 + a_SS1*A_S1 + a_BS2*A_B2 + a_SS2*A_S2
        lam_max = lam_B + lam_S

        # draw next candidate event time
        dt = -np.log(np.random.random()) / lam_max
        t += dt

        if t >= T_max:
            break

        # decay sums
        d1 = np.exp(-beta1 * dt)
        d2 = np.exp(-beta2 * dt)
        A_B1 *= d1;  A_S1 *= d1
        A_B2 *= d2;  A_S2 *= d2

        # thinning: accept or reject
        lam_B_new = mu_B + a_BB1*A_B1 + a_SB1*A_S1 + a_BB2*A_B2 + a_SB2*A_S2
        lam_S_new = mu_S + a_BS1*A_B1 + a_SS1*A_S1 + a_BS2*A_B2 + a_SS2*A_S2
        u = np.random.random() * lam_max

        if u < lam_B_new:       # BUY event
            T_B.append(t)
            A_B1 += 1.0;  A_B2 += 1.0
        elif u < lam_B_new + lam_S_new:  # SELL event
            T_S.append(t)
            A_S1 += 1.0;  A_S2 += 1.0
        # else: rejected

    return np.array(T_B), np.array(T_S)

# known ground truth
TRUE = dict(mu_B=0.3, mu_S=0.25,
            a_BB1=70.0, a_SS1=80.0, a_BS1=1.0, a_SB1=2.0,  # fast: phi=0.7,0.8,0.01,0.02
            a_BB2=0.04, a_SS2=0.04, a_BS2=0.015, a_SB2=0.025,  # slow
            beta1=100.0, beta2=1.0)

np.random.seed(42)
T_B_syn, T_S_syn = simulate_bivariate_soe(
    TRUE['mu_B'], TRUE['mu_S'],
    TRUE['a_BB1'], TRUE['a_SS1'], TRUE['a_BS1'], TRUE['a_SB1'],
    TRUE['a_BB2'], TRUE['a_SS2'], TRUE['a_BS2'], TRUE['a_SB2'],
    TRUE['beta1'], TRUE['beta2'], T_max=3600.0)

print(f"Synthetic: n_B={len(T_B_syn)}, n_S={len(T_S_syn)}")
rec = fit_window_soe_fast(T_B_syn, T_S_syn, beta1=100.0, beta2=1.0)
print("\nTrue vs Recovered:")
print(f"phi_BB1: true={TRUE['a_BB1']/100:.3f}  recovered={rec['phi_BB1']:.3f}")
print(f"phi_SS1: true={TRUE['a_SS1']/100:.3f}  recovered={rec['phi_SS1']:.3f}")
print(f"phi_BB2: true={TRUE['a_BB2']/1:.3f}  recovered={rec['phi_BB2']:.3f}")
print(f"phi_SB2: true={TRUE['a_SB2']/1:.3f}  recovered={rec['phi_SB2']:.3f}")
print(f"eta:     true~0.89       recovered={rec['eta']:.3f}")

Synthetic: n_B=5407, n_S=6905

True vs Recovered:
phi_BB1: true=0.700  recovered=0.699
phi_SS1: true=0.800  recovered=0.809
phi_BB2: true=0.040  recovered=0.036
phi_SB2: true=0.025  recovered=0.032
eta:     true~0.89       recovered=0.860


In [71]:
# ── Unit tests ──────────────────────────────────────────────────────
def run_tests():
    print("Running tests...")

    # 1. Synthetic parameter recovery
    np.random.seed(42)
    T_B_t, T_S_t = simulate_bivariate_soe(
        0.3, 0.25, 70.0, 80.0, 1.0, 2.0,
        0.04, 0.04, 0.015, 0.025,
        100.0, 1.0, 3600.0)
    r = fit_window_soe_fast(T_B_t, T_S_t, beta1=100.0, beta2=1.0)
    assert r is not None, "fit returned None on synthetic data"
    assert abs(r['phi_BB1'] - 0.7) < 0.15, f"phi_BB1 off: {r['phi_BB1']:.3f}"
    assert abs(r['phi_SS1'] - 0.8) < 0.15, f"phi_SS1 off: {r['phi_SS1']:.3f}"
    assert r['eta'] < 1.0,                  f"unstable: eta={r['eta']:.3f}"
    print("  ✓ synthetic recovery")

    # 2. Stability on real first window
    t0 = btc['t'].min()
    T_B = btc_buy [(btc_buy  >= t0) & (btc_buy  < t0+3600)] - t0
    T_S = btc_sell[(btc_sell >= t0) & (btc_sell < t0+3600)] - t0
    r2 = fit_window_soe_fast(T_B, T_S)
    assert r2 is not None,      "fit returned None on real data"
    assert r2['eta'] < 1.0,     f"unstable on real data: eta={r2['eta']:.3f}"
    assert r2['phi_BB1'] > 0.3, f"phi_BB1 suspiciously low: {r2['phi_BB1']:.3f}"
    assert r2['loglik'] > 0,    f"loglik negative: {r2['loglik']:.0f}"
    print("  ✓ real data stability")

    # 3. Data integrity
    assert (btc['event_type'].isin([0,1])).all(), "unexpected event types in BTC"
    assert btc['timestamp_us'].is_monotonic_increasing, "timestamps not sorted"
    assert len(btc_buy) > 1_000_000, "unexpectedly few BTC buy events"
    print("  ✓ data integrity")

    print("All tests passed.")

run_tests()

Running tests...
  ✓ synthetic recovery
  ✓ real data stability
  ✓ data integrity
All tests passed.


In [72]:
from scipy.stats import ks_1samp, expon

def time_rescaling_test(T_B, T_S, result, ax, process='buy', title='', beta1=100.0, beta2=1.0):
    inc = compute_compensator_increments(T_B, T_S, result, process, beta1, beta2)
    ks_d, ks_p = ks_1samp(inc, expon.cdf)
    print(f"  {title} {process.upper():4s}  n={len(inc):,}  mean={inc.mean():.4f}  "
          f"KS={ks_d:.4f}  p={ks_p:.2e}")

    q  = np.linspace(0.005, 0.995, min(len(inc), 1500))
    th = expon.ppf(q);  em = np.quantile(inc, q)
    lim = max(th.max(), em.max()) * 1.05
    ax.scatter(th, em, alpha=0.4, s=15, label='Empirical')
    ax.plot([0, lim], [0, lim], 'r--', lw=2, label='Perfect fit')
    ax.set_xlim(0, lim);  ax.set_ylim(0, lim)
    ax.set_xlabel('Theoretical Exp(1) quantiles', fontsize=11)
    ax.set_ylabel('Empirical compensator increments', fontsize=11)
    ax.set_title(f'{title} — {process.upper()} process', fontsize=11, fontweight='bold')
    ax.legend();  ax.grid(True, alpha=0.3)
    ax.text(0.97, 0.05, f'KS = {ks_d:.4f}\np  = {ks_p:.2e}\nmean = {inc.mean():.3f}',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.85))
    return inc
    
def aggregate_sweeps(timestamps_us: np.ndarray, gap_us: int = 100) -> np.ndarray:
    """
    Collapse consecutive same-direction aggTrade events within `gap_us`
    microseconds into a single event, keeping the LATEST timestamp of each burst.
    Call separately for the buy and sell streams.

    gap_us=100 is the empirically optimal threshold for BTC/USDT (minimises KS
    statistic across thresholds 50–5000µs). A single market order sweeping
    multiple price levels on Binance completes in under 100µs.
    """
    ts = np.sort(timestamps_us.astype(np.int64))
    if len(ts) == 0:
        return ts
    is_last = np.empty(len(ts), dtype=bool)
    is_last[:-1] = np.diff(ts) > gap_us
    is_last[-1]  = True
    return ts[is_last]


def compute_compensator_increments(T_B, T_S, result, process='buy', beta1=100.0, beta2=1.0):
    """
    process='buy'  → Λ_B increments, uses μ_B, φ_BB, φ_SB
    process='sell' → Λ_S increments, uses μ_S, φ_SS, φ_BS
    Under a correctly-specified model, increments ~ i.i.d. Exp(1).
    """
    betas = np.array([beta1, beta2])

    if process == 'buy':
        mu        = result['mu_B']
        phi_self  = np.array([result['phi_BB1'], result['phi_BB2']])
        phi_cross = np.array([result['phi_SB1'], result['phi_SB2']])
        target    = 0.0
    else:
        mu        = result['mu_S']
        phi_self  = np.array([result['phi_SS1'], result['phi_SS2']])
        phi_cross = np.array([result['phi_BS1'], result['phi_BS2']])
        target    = 1.0

    all_t = np.concatenate([T_B, T_S])
    all_e = np.concatenate([np.zeros(len(T_B)), np.ones(len(T_S))])
    order = np.argsort(all_t, kind='stable')
    all_t = all_t[order];  all_e = all_e[order]

    increments, A_self, A_cross = [], np.zeros(2), np.zeros(2)
    last_t = in_interval = comp = 0.0

    for k in range(len(all_t)):
        t, etype, dt = all_t[k], all_e[k], all_t[k] - last_t
        if dt > 0.0:
            e = np.exp(-betas * dt)
            if in_interval:
                comp += mu * dt + np.dot(phi_self,  A_self  * (1.0 - e)) \
                                + np.dot(phi_cross, A_cross * (1.0 - e))
            A_self *= e;  A_cross *= e
        if etype == target:
            if in_interval:
                increments.append(comp)
            comp = 0.0;  in_interval = True;  A_self += 1.0
        else:
            A_cross += 1.0
        last_t = t

    return np.array(increments)


print("✓ aggregate_sweeps | compute_compensator_increments")

✓ aggregate_sweeps | compute_compensator_increments


In [82]:

thresholds_us = [50, 100, 150, 200, 250, 300, 400, 500, 750, 1000, 2000, 5000]
ks_stats, n_events = [], []
t0_btc = int(btc['t'].min() * 1e6)
raw_B  = btc.loc[btc['event_type'] == 0, 'timestamp_us'].values
raw_S  = btc.loc[btc['event_type'] == 1, 'timestamp_us'].values
t1_btc = t0_btc + 3_600_000_000
raw_B  = raw_B[(raw_B >= t0_btc) & (raw_B < t1_btc)]
raw_S  = raw_S[(raw_S >= t0_btc) & (raw_S < t1_btc)]

np.random.seed(0)
for gap in thresholds_us:
    agg_B = (aggregate_sweeps(raw_B, gap_us=gap) - t0_btc) / 1e6
    agg_S = (aggregate_sweeps(raw_S, gap_us=gap) - t0_btc) / 1e6
    res   = fit_window_soe_fast(agg_B, agg_S)
    if res is None:
        ks_stats.append(np.nan);  n_events.append(np.nan);  continue
    vals  = compute_compensator_increments(agg_B, agg_S, res)
    ks_stats.append(ks_1samp(vals, expon.cdf)[0])
    n_events.append(len(agg_B))
    print(f"  gap={gap:>6}µs  n_B={len(agg_B):>6,}  KS={ks_stats[-1]:.4f}")

GAP_US = thresholds_us[int(np.nanargmin(ks_stats))]
print(f"\n→ optimal threshold: {GAP_US}µs")

fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()

ax1.plot([t/1000 for t in thresholds_us], ks_stats,
         'o-', color='steelblue', lw=2, label='KS stat')
ax2.plot([t/1000 for t in thresholds_us], n_events,
         's--', color='darkorange', lw=1.5, label='n_B events')

opt_idx = int(np.nanargmin(ks_stats))
opt_ks  = ks_stats[opt_idx]

ax1.set_xscale('log')
ax1.set_xlabel('Sweep threshold (log scale)', fontsize=11)
ax1.set_ylabel('KS statistic', fontsize=11, color='steelblue', labelpad=14)
ax2.set_ylabel('BUY events after aggregation', fontsize=11, color='darkorange', labelpad=14)
ax1.tick_params(axis='y', labelcolor='steelblue')
ax2.tick_params(axis='y', labelcolor='darkorange')

ax1.set_xticks([t/1000 for t in thresholds_us])
ax1.set_xticklabels([f'{t}µs' for t in thresholds_us], rotation=40, ha='right')
ax1.set_title(
    f'GOF vs sweep threshold — BTC first hour\n'
    f'Optimal threshold: {thresholds_us[opt_idx]}µs  (KS = {opt_ks:.4f})',
    fontsize=11, fontweight='bold', linespacing=1.6
)

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc='upper right', fontsize=9, framealpha=0.9)

ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/ks_vs_threshold.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: results/ks_vs_threshold.png")

  gap=    50µs  n_B=11,248  KS=0.0212
  gap=   100µs  n_B=11,154  KS=0.0205
  gap=   150µs  n_B=11,110  KS=0.0205
  gap=   200µs  n_B=11,074  KS=0.0206
  gap=   250µs  n_B=11,058  KS=0.0215
  gap=   300µs  n_B=11,046  KS=0.0224
  gap=   400µs  n_B=11,028  KS=0.0236
  gap=   500µs  n_B=11,012  KS=0.0242
  gap=   750µs  n_B=10,960  KS=0.0276
  gap=  1000µs  n_B=10,845  KS=0.0307
  gap=  2000µs  n_B=10,405  KS=0.0389
  gap=  5000µs  n_B= 9,479  KS=0.0416

→ optimal threshold: 100µs
✓ Saved: results/ks_vs_threshold.png


/var/folders/x_/4gqtnjds52d1_36fmtw0bv2h0000gn/T/ipykernel_38085/3448216695.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [74]:
import os, time

def _qq_panel(ax, increments, title, ks_d, ks_p, color):
    q  = np.linspace(0.005, 0.995, min(len(increments), 1500))
    th = expon.ppf(q);  em = np.quantile(increments, q)
    lim = max(th.max(), em.max()) * 1.05
    ax.scatter(th, em, alpha=0.40, s=12, color=color, label='Empirical')
    ax.plot([0, lim], [0, lim], 'r--', lw=2, label='y = x  (perfect fit)')
    ax.set_xlim(0, lim);  ax.set_ylim(0, lim)
    ax.set_xlabel('Theoretical Exp(1) quantiles', fontsize=11)
    ax.set_ylabel('Empirical compensator increments', fontsize=11)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9);  ax.grid(True, alpha=0.3)
    ax.text(0.97, 0.05,
            f'KS = {ks_d:.4f}\np  = {ks_p:.2e}\nmean = {increments.mean():.3f}',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.85))

def _first_hour_us(df, t0_us):
    t1_us = t0_us + 3_600_000_000
    b = df.loc[df['event_type'] == 0, 'timestamp_us'].values
    s = df.loc[df['event_type'] == 1, 'timestamp_us'].values
    return b[(b >= t0_us) & (b < t1_us)], s[(s >= t0_us) & (s < t1_us)]

sym_data = {
    'BTC': (btc, int(btc['t'].min() * 1e6)),
    'ETH': (eth, int(eth['t'].min() * 1e6)),
}

streams = {}
for sym, (df, t0_us) in sym_data.items():
    raw_B, raw_S = _first_hour_us(df, t0_us)
    streams[sym] = dict(
        raw_B=raw_B, raw_S=raw_S,
        agg_B=aggregate_sweeps(raw_B, GAP_US),
        agg_S=aggregate_sweeps(raw_S, GAP_US),
        t0_us=t0_us,
    )
    print(f"{sym}  raw n_B={len(raw_B):,}  →  agg n_B={len(streams[sym]['agg_B']):,}  "
          f"({100*(1-len(streams[sym]['agg_B'])/len(raw_B)):.1f}% reduction)")

# ── Baseline GOF (raw streams, BUY + SELL, both symbols) ─────────────────────
np.random.seed(0)
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
fig.suptitle('Time-Rescaling GOF — RAW baseline, first 1-hour window',
             fontsize=13, fontweight='bold')

for row, (sym, s) in enumerate(streams.items()):
    t0  = s['t0_us']
    T_B = (s['raw_B'] - t0) / 1e6
    T_S = (s['raw_S'] - t0) / 1e6
    res = fit_window_soe_fast(T_B, T_S)
    for col, process in enumerate(['buy', 'sell']):
        time_rescaling_test(T_B, T_S, res, ax=axes[row, col],
                                      process=process, title=sym)

plt.tight_layout()
os.makedirs('results', exist_ok=True)
plt.savefig('results/gof_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Fit raw and aggregated models ─────────────────────────────────────────────
fits = {sym: {} for sym in sym_data}
np.random.seed(0)
for sym, s in streams.items():
    t0 = s['t0_us']
    for variant, (us_B, us_S) in [('raw', (s['raw_B'], s['raw_S'])),
                                   ('agg', (s['agg_B'], s['agg_S']))]:
        T_B = (us_B - t0) / 1e6;  T_S = (us_S - t0) / 1e6
        tic = time.time()
        fits[sym][variant] = fit_window_soe_fast(T_B, T_S)
        print(f"{sym} {variant:3s}  {time.time()-tic:.1f}s  "
              f"μ_B={fits[sym][variant]['mu_B']:.4f}  η={fits[sym][variant]['eta']:.4f}")

# ── Compensator increments and KS stats ───────────────────────────────────────
inc = {sym: {} for sym in sym_data}
ks  = {sym: {} for sym in sym_data}
for sym, s in streams.items():
    t0 = s['t0_us']
    for variant, (us_B, us_S) in [('raw', (s['raw_B'], s['raw_S'])),
                                   ('agg', (s['agg_B'], s['agg_S']))]:
        T_B = (us_B - t0) / 1e6;  T_S = (us_S - t0) / 1e6
        inc[sym][variant] = compute_compensator_increments(T_B, T_S, fits[sym][variant])
        ks[sym][variant]  = ks_1samp(inc[sym][variant], expon.cdf)

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"\n{'Symbol':<6} {'Variant':<8} {'n_B':>7} {'mean':>7} {'KS':>8} {'p':>12} {'reduction':>10}")
print("─" * 62)
for sym, s in streams.items():
    for variant, us_B in [('raw', s['raw_B']), ('agg', s['agg_B'])]:
        n, mean  = len(inc[sym][variant]), inc[sym][variant].mean()
        d, p     = ks[sym][variant]
        red      = (f"{100*(ks[sym]['raw'][0]-d)/ks[sym]['raw'][0]:.1f}%"
                    if variant == 'agg' else '—')
        print(f"{sym:<6} {variant:<8} {n:>7,} {mean:>7.4f} {d:>8.4f} {p:>12.2e} {red:>10}")

# ── 2×2 sweep comparison QQ ───────────────────────────────────────────────────
colors = {'raw': 'steelblue', 'agg': 'darkorange'}
labels = {'raw': 'RAW', 'agg': f'AGG {GAP_US}µs'}

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
fig.suptitle(f'Sweep aggregation GOF — {GAP_US}µs threshold\n'
             'BUY process, first 1-hour window', fontsize=13, fontweight='bold')

for row, sym in enumerate(['BTC', 'ETH']):
    s = streams[sym]
    for col, variant in enumerate(['raw', 'agg']):
        d, p = ks[sym][variant]
        _qq_panel(axes[row, col], inc[sym][variant],
                  f'{sym} {labels[variant]}  (n_B={len(s[f"{variant}_B"]):,})',
                  d, p, colors[variant])

plt.tight_layout()
plt.savefig('results/gof_sweep_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: results/gof_sweep_comparison.png")

BTC  raw n_B=30,385  →  agg n_B=11,154  (63.3% reduction)
ETH  raw n_B=32,493  →  agg n_B=13,371  (58.8% reduction)
  BTC BUY   n=30,384  mean=1.0002  KS=0.6195  p=0.00e+00
  BTC SELL  n=31,625  mean=1.0004  KS=0.6966  p=0.00e+00
  ETH BUY   n=32,492  mean=0.9990  KS=0.5765  p=0.00e+00
  ETH SELL  n=36,685  mean=0.9980  KS=0.6446  p=0.00e+00


/var/folders/x_/4gqtnjds52d1_36fmtw0bv2h0000gn/T/ipykernel_38085/4121865030.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


BTC raw  2.5s  μ_B=1.4186  η=0.8901
BTC agg  2.2s  μ_B=0.8624  η=0.7532
ETH raw  3.0s  μ_B=0.9276  η=0.9309
ETH agg  1.1s  μ_B=0.7300  η=0.8446

Symbol Variant      n_B    mean       KS            p  reduction
──────────────────────────────────────────────────────────────
BTC    raw       30,384  1.0002   0.6195     0.00e+00          —
BTC    agg       11,153  0.9994   0.0205     1.69e-04      96.7%
ETH    raw       32,492  1.0033   0.5765     0.00e+00          —
ETH    agg       13,370  0.9998   0.0256     5.14e-08      95.6%
✓ Saved: results/gof_sweep_comparison.png


/var/folders/x_/4gqtnjds52d1_36fmtw0bv2h0000gn/T/ipykernel_38085/4121865030.py:115: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [75]:
print("Rolling calibration")
!pip install tqdm
import pandas as pd
from tqdm import tqdm

def rolling_calibration(buy_arr, sell_arr, symbol, 
                        beta1=100.0, beta2=1.0,
                        window_sec=3600):
    t_start = buy_arr[0] if buy_arr[0] < sell_arr[0] else sell_arr[0]
    t_end   = buy_arr[-1] if buy_arr[-1] > sell_arr[-1] else sell_arr[-1]

    windows = np.arange(t_start, t_end - window_sec, window_sec)
    results = []

    for t0 in tqdm(windows, desc=symbol):
        t1 = t0 + window_sec
        T_B = buy_arr [(buy_arr  >= t0) & (buy_arr  < t1)] - t0
        T_S = sell_arr[(sell_arr >= t0) & (sell_arr < t1)] - t0

        r = fit_window_soe_fast(T_B, T_S, beta1=beta1, beta2=beta2)
        if r is None:
            continue

        utc_hour = int((t0 % 86400) / 3600)
        results.append({
            'window_start': t0,
            'utc_hour':     utc_hour,
            'symbol':       symbol,
            'n_B':          len(T_B),
            'n_S':          len(T_S),
            **r
        })

    return pd.DataFrame(results)

# # run both symbols
df_btc = rolling_calibration(btc_buy, btc_sell, 'BTCUSDT')
df_eth = rolling_calibration(eth_buy, eth_sell, 'ETHUSDT')

# # save
import os
os.makedirs('results', exist_ok=True)
df_btc.to_csv('results/results_btcusdt.csv', index=False)
df_eth.to_csv('results/results_ethusdt.csv', index=False)

print(f"BTC: {len(df_btc)} windows")
print(f"ETH: {len(df_eth)} windows")
print(df_btc[['utc_hour','phi_BB1','phi_SS1','phi_BB2','phi_SS2',
              'phi_SB2','phi_BS2','eta']].head(5))

Rolling calibration


In [76]:
# ── C2: UTC seasonality — KW global test + targeted UTC-13 contrast ───────────
import pandas as pd
from scipy.stats import kruskal, mannwhitneyu

df_btc = pd.read_csv('results/results_btcusdt.csv')
df_eth = pd.read_csv('results/results_ethusdt.csv')

variables = {
    'eta':     'η (branching ratio)',
    'phi_BB1': 'φ_BB1 (fast BUY self-excit.)',
    'phi_SS1': 'φ_SS1 (fast SELL self-excit.)',
    'phi_SB2': 'φ_SB2 (slow SELL→BUY)',
    'phi_BS2': 'φ_BS2 (slow BUY→SELL)',
}

print(f"{'Variable':<12} {'Asset':<5} {'med_13':>7} {'med_rest':>9} "
      f"{'Δ%':>6} {'KW_p':>8} {'MW_p':>8} {'':>10}")
print("─" * 72)

for var in variables:
    for sym, df in [('BTC', df_btc), ('ETH', df_eth)]:
        # KW: all 24 hours
        groups = [g[var].dropna().values for _, g in df.groupby('utc_hour')
                  if len(g[var].dropna()) > 0]
        _, kw_p = kruskal(*groups)

        # MW: UTC 13 vs rest
        h13   = df[df['utc_hour'] == 13][var].dropna().values
        hrest = df[df['utc_hour'] != 13][var].dropna().values
        _, mw_p = mannwhitneyu(h13, hrest, alternative='greater')

        delta = 100 * (h13.mean() - hrest.mean()) / hrest.mean()
        sig   = '✓' if mw_p < 0.05 else ('~' if mw_p < 0.10 else '✗')
        print(f"{var:<12} {sym:<5} {h13.mean():>7.4f} {hrest.mean():>9.4f} "
              f"{delta:>+6.1f}% {kw_p:>8.4f} {mw_p:>8.4f} {sig:>10}")
    print()

print(f"UTC-13 obs: {len(h13)}  |  Rest obs: {len(hrest)}  |  Total windows per asset: {len(df_btc)}")
print("KW: H₀ = all 24 hours identical (global, low power at 6 obs/hour)")
print("MW: H₁ = UTC-13 > rest (targeted one-sided, higher power)")

Variable     Asset  med_13  med_rest     Δ%     KW_p     MW_p           
────────────────────────────────────────────────────────────────────────
eta          BTC    0.8339    0.7420  +12.4%   0.1452   0.0035          ✓
eta          ETH    0.8930    0.8153   +9.5%   0.1294   0.0006          ✓

phi_BB1      BTC    0.7482    0.6291  +18.9%   0.0844   0.0015          ✓
phi_BB1      ETH    0.7812    0.7134   +9.5%   0.0030   0.0035          ✓

phi_SS1      BTC    0.7716    0.6851  +12.6%   0.3666   0.0059          ✓
phi_SS1      ETH    0.8024    0.7497   +7.0%   0.6230   0.0196          ✓

phi_SB2      BTC    0.0184    0.0115  +60.3%   0.7836   0.0424          ✓
phi_SB2      ETH    0.0251    0.0116 +115.5%   0.1737   0.0010          ✓

phi_BS2      BTC    0.0131    0.0104  +25.9%   0.8132   0.1429          ✗
phi_BS2      ETH    0.0175    0.0113  +55.1%   0.4172   0.0280          ✓

UTC-13 obs: 6  |  Rest obs: 139  |  Total windows per asset: 144
KW: H₀ = all 24 hours identical (global, low

In [77]:
# ── C2 Option 2: Regression + bootstrap F-test for UTC seasonality ─────────────
import numpy as np
import pandas as pd
from scipy.stats import f as f_dist

df_btc = pd.read_csv('results/results_btcusdt.csv')
df_eth = pd.read_csv('results/results_ethusdt.csv')
df_btc['asset'] = 0;  df_eth['asset'] = 1
df = pd.concat([df_btc, df_eth], ignore_index=True)

variables = {
    'eta':     'η',
    'phi_BB1': 'φ_BB1',
    'phi_SS1': 'φ_SS1',
    'phi_SB2': 'φ_SB2',
    'phi_BS2': 'φ_BS2',
}

def build_X(d):
    n = len(d)
    H = np.column_stack([(d['utc_hour'].values == h).astype(float) for h in range(1, 24)])
    return np.hstack([np.ones((n, 1)), H, d['asset'].values.reshape(-1, 1)])

def ols_F(d, var):
    y = d[var].values;  n = len(y)
    X = build_X(d);     k = X.shape[1]
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    ss_res = np.sum((y - X @ beta)**2)
    r2 = 1 - ss_res / np.sum((y - y.mean())**2)
    # restricted: intercept + asset only
    Xr = np.hstack([np.ones((n, 1)), d['asset'].values.reshape(-1, 1)])
    ss_r = np.sum((y - Xr @ np.linalg.lstsq(Xr, y, rcond=None)[0])**2)
    F = ((ss_r - ss_res) / 23) / (ss_res / (n - k))
    return r2, F, 1 - f_dist.cdf(F, 23, n - k), n, k

def bootstrap_F_p(d, var, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    y = d[var].values;  n = len(y)
    Xr = np.hstack([np.ones((n, 1)), d['asset'].values.reshape(-1, 1)])
    beta_r = np.linalg.lstsq(Xr, y, rcond=None)[0]
    resid_r = y - Xr @ beta_r
    yhat_r  = Xr @ beta_r
    _, F_obs, _, _, _ = ols_F(d, var)
    F_boots = []
    for _ in range(n_boot):
        db = d.copy()
        db[var] = yhat_r + rng.choice(resid_r, size=n, replace=True)
        _, Fb, _, _, _ = ols_F(db, var)
        F_boots.append(Fb)
    return np.mean(np.array(F_boots) >= F_obs)

print("OLS: var ~ 23 hour dummies + asset dummy")
print("F-test H₀: all 23 hour coefficients = 0 (no UTC seasonality)")
print(f"\n{'Variable':<10} {'R²':>6} {'F-stat':>8} {'p(OLS)':>9} {'p(boot)':>9} {'':>10}")
print("─" * 55)

for var, label in variables.items():
    r2, F, p_ols, n, k = ols_F(df, var)
    p_boot = bootstrap_F_p(df, var)
    sig = '✓ p<0.05' if p_boot < 0.05 else ('~ p<0.10' if p_boot < 0.10 else '✗ n.s.')
    print(f"{var:<10} {r2:>6.3f} {F:>8.2f} {p_ols:>9.4f} {p_boot:>9.4f} {sig:>10}")

print(f"\nn={n} obs ({len(df_btc)} BTC + {len(df_eth)} ETH), "
      f"df_resid={n-k}, n_boot=2000")

OLS: var ~ 23 hour dummies + asset dummy
F-test H₀: all 23 hour coefficients = 0 (no UTC seasonality)

Variable       R²   F-stat    p(OLS)   p(boot)           
───────────────────────────────────────────────────────
eta         0.352     2.56    0.0002    0.0010   ✓ p<0.05
phi_BB1     0.394     3.52    0.0000    0.0000   ✓ p<0.05
phi_SS1     0.251     1.72    0.0240    0.0265   ✓ p<0.05
phi_SB2     0.111     1.43    0.0951    0.0965   ~ p<0.10
phi_BS2     0.084     1.02    0.4452    0.4410     ✗ n.s.

n=289 obs (144 BTC + 145 ETH), df_resid=264, n_boot=2000


In [78]:
# ── Realized volatility per window and correlation with η ─────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

df_btc = pd.read_csv('results/results_btcusdt.csv')
df_eth = pd.read_csv('results/results_ethusdt.csv')

def compute_rv(df_raw, window_starts, window_sec=3600):
    """Realized volatility = sqrt(sum of squared log-returns) per window."""
    trades = df_raw[df_raw['event_type'].isin([0, 1])].sort_values('t')
    prices = trades['price'].values
    times  = trades['t'].values
    rvs = []
    for t0 in window_starts:
        p = prices[(times >= t0) & (times < t0 + window_sec)]
        if len(p) < 2:
            rvs.append(np.nan)
        else:
            rvs.append(np.sqrt(np.sum(np.diff(np.log(p))**2)))
    return np.array(rvs)

df_btc['rv'] = compute_rv(btc, df_btc['window_start'].values)
df_eth['rv'] = compute_rv(eth, df_eth['window_start'].values)

print(f"{'Asset':<6} {'Spearman ρ':>11} {'p-value':>10} {'n':>5}")
print("─" * 38)
for sym, df in [('BTC', df_btc), ('ETH', df_eth)]:
    mask = ~np.isnan(df['rv'])
    rho, p = spearmanr(df.loc[mask, 'eta'], df.loc[mask, 'rv'])
    print(f"{sym:<6} {rho:>11.3f} {p:>10.4f} {mask.sum():>5}")

# Scatter plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Branching ratio η vs Realized Volatility per 1-hour window',
             fontsize=12, fontweight='bold')
colors = {'BTC': 'steelblue', 'ETH': 'darkorange'}

for ax, (sym, df) in zip(axes, [('BTC', df_btc), ('ETH', df_eth)]):
    mask = ~np.isnan(df['rv'])
    x = df.loc[mask, 'eta'].values
    y = df.loc[mask, 'rv'].values
    rho, p = spearmanr(x, y)
    ax.scatter(x, y, alpha=0.4, s=20, color=colors[sym])
    m, b = np.polyfit(x, y, 1)
    xl = np.linspace(x.min(), x.max(), 100)
    ax.plot(xl, m*xl + b, 'r-', lw=2, alpha=0.8)
    ax.set_xlabel('η (branching ratio)', fontsize=11)
    ax.set_ylabel('Realized Volatility', fontsize=11)
    ax.set_title(f'{sym}  ρ={rho:.3f}  p={p:.4f}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs('results', exist_ok=True)
plt.savefig('results/eta_vs_rv.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: results/eta_vs_rv.png")

Asset   Spearman ρ    p-value     n
──────────────────────────────────────
BTC          0.840     0.0000   144
ETH          0.818     0.0000   145
✓ Saved: results/eta_vs_rv.png


/var/folders/x_/4gqtnjds52d1_36fmtw0bv2h0000gn/T/ipykernel_38085/3051599161.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [83]:
# GOF robustness across 12 sampled windows ─────────────────────────────
# One window per 2-hour UTC bucket, drawn from df_btc.
# Tests whether KS ≈ 0.02 is specific to the first window or dataset-wide.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_1samp, expon

df_btc = pd.read_csv('results/results_btcusdt.csv')

# Sample one window per 2-hour UTC bucket (buckets: 0-1, 2-3, ..., 22-23)
np.random.seed(42)
sampled = []
for bucket_start in range(0, 24, 2):
    bucket_hours = [bucket_start, bucket_start + 1]
    pool = df_btc[df_btc['utc_hour'].isin(bucket_hours)]
    if len(pool) > 0:
        sampled.append(pool.sample(1).iloc[0])
sampled = pd.DataFrame(sampled).reset_index(drop=True)
print(f"Sampled {len(sampled)} windows across UTC buckets")
print(sampled[['window_start', 'utc_hour', 'n_B', 'n_S']].to_string(index=False))

# For each sampled window, run sweep aggregation + GOF
ks_raw_list, ks_agg_list, utc_hours = [], [], []

for _, row in sampled.iterrows():
    t0 = row['window_start']
    t1 = t0 + 3600

    raw_us_B = btc.loc[btc['event_type'] == 0, 'timestamp_us'].values
    raw_us_S = btc.loc[btc['event_type'] == 1, 'timestamp_us'].values
    t0_us = int(t0 * 1e6);  t1_us = int(t1 * 1e6)

    raw_B = raw_us_B[(raw_us_B >= t0_us) & (raw_us_B < t1_us)]
    raw_S = raw_us_S[(raw_us_S >= t0_us) & (raw_us_S < t1_us)]
    agg_B = aggregate_sweeps(raw_B, GAP_US)
    agg_S = aggregate_sweeps(raw_S, GAP_US)

    T_B_raw = (raw_B - t0_us) / 1e6;  T_S_raw = (raw_S - t0_us) / 1e6
    T_B_agg = (agg_B - t0_us) / 1e6;  T_S_agg = (agg_S - t0_us) / 1e6

    res_raw = fit_window_soe_fast(T_B_raw, T_S_raw)
    res_agg = fit_window_soe_fast(T_B_agg, T_S_agg)

    if res_raw and res_agg:
        inc_raw = compute_compensator_increments(T_B_raw, T_S_raw, res_raw)
        inc_agg = compute_compensator_increments(T_B_agg, T_S_agg, res_agg)
        ks_raw_list.append(ks_1samp(inc_raw, expon.cdf)[0])
        ks_agg_list.append(ks_1samp(inc_agg, expon.cdf)[0])
        utc_hours.append(int(row['utc_hour']))
        print(f"  UTC {int(row['utc_hour']):02d}  raw KS={ks_raw_list[-1]:.4f}  "
              f"agg KS={ks_agg_list[-1]:.4f}  "
              f"reduction={100*(ks_raw_list[-1]-ks_agg_list[-1])/ks_raw_list[-1]:.1f}%")

print(f"\nRaw  KS: mean={np.mean(ks_raw_list):.4f}  range=[{min(ks_raw_list):.4f}, {max(ks_raw_list):.4f}]")
print(f"Agg  KS: mean={np.mean(ks_agg_list):.4f}  range=[{min(ks_agg_list):.4f}, {max(ks_agg_list):.4f}]")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f'GOF robustness — 100µs sweep aggregation across 12 sampled BTC windows\n'
             f'(one per 2-hour UTC bucket)', fontsize=11, fontweight='bold')

for ax, vals, label, color in [
    (axes[0], ks_raw_list,  'RAW stream',          'steelblue'),
    (axes[1], ks_agg_list,  f'AGG {GAP_US}µs stream', 'darkorange'),
]:
    ax.hist(vals, bins=8, color=color, alpha=0.75, edgecolor='white')
    ax.axvline(np.mean(vals), color='red', lw=2, linestyle='--',
               label=f'mean = {np.mean(vals):.4f}')
    ax.set_xlabel('KS statistic', fontsize=11)
    ax.set_ylabel('Count (windows)', fontsize=11)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs('results', exist_ok=True)
plt.savefig('results/gof_robustness.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: results/gof_robustness.png")

Sampled 12 windows across UTC buckets
 window_start  utc_hour   n_B   n_S
 1.778892e+09         0  6729  6520
 1.778899e+09         2  6104  5288
 1.778737e+09         5 15167 13055
 1.778830e+09         7 14410 14905
 1.778488e+09         8  8722  6802
 1.778841e+09        10 12861 10683
 1.778589e+09        12 18141 17366
 1.778596e+09        14 29565 28452
 1.778521e+09        17 20024 16595
 1.778784e+09        18 17276 18510
 1.778794e+09        21  9761  7937
 1.778715e+09        23 12983  7971
  UTC 00  raw KS=0.3828  agg KS=0.0189  reduction=95.1%
  UTC 02  raw KS=0.3964  agg KS=0.0162  reduction=95.9%
  UTC 05  raw KS=0.6460  agg KS=0.0338  reduction=94.8%
  UTC 07  raw KS=0.4402  agg KS=0.0380  reduction=91.4%
  UTC 08  raw KS=0.5424  agg KS=0.0172  reduction=96.8%
  UTC 10  raw KS=0.4670  agg KS=0.0312  reduction=93.3%
  UTC 12  raw KS=0.5590  agg KS=0.0184  reduction=96.7%
  UTC 14  raw KS=0.5980  agg KS=0.0261  reduction=95.6%
  UTC 17  raw KS=0.6163  agg KS=0.0252  reduct

/var/folders/x_/4gqtnjds52d1_36fmtw0bv2h0000gn/T/ipykernel_38085/399172865.py:79: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [79]:
def plot_kernel_norms_by_hour(df_btc, df_eth):
    hours = np.arange(24)

    def get_medians(df, col):
        grouped = df.groupby('utc_hour')[col].median()
        return [grouped.loc[h] if h in grouped.index else np.nan for h in hours]

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle(
        'Hawkes Kernel Norms: BTC/USDT vs ETH/USDT',
        fontsize=15,
        fontweight='bold',
        y=0.98
    )
    
    fig.text(
        0.5, 0.945,
        'Median across 6 daily observations, May 2026; U.S. DST active: EDT = UTC−4',
        ha='center',
        fontsize=10,
        color='dimgray'
    )
    # Market session lines: (x_position, color, linestyle, short_label)
    market_lines = [
        (7,    '#2196F3', '--', 'LSE 07:00'),
        (8,    '#9C27B0', '--', 'NYSE pre 08:00'),
        (13.5, '#F44336', '-',  'NYSE open 13:30'),
        (22,   '#FF9800', '--', 'Asia 22:00'),
    ]

    panels = [
        ('phi_BB1', 'φ_BB1 — Buy self-excitation (fast, β=100)',      axes[0, 0]),
        ('phi_SS1', 'φ_SS1 — Sell self-excitation (fast, β=100)',     axes[0, 1]),
        ('phi_SB2', 'φ_SB2 — SELL→BUY cross-excitation (slow, β=1)', axes[1, 0]),
        ('phi_BS2', 'φ_BS2 — BUY→SELL cross-excitation (slow, β=1)', axes[1, 1]),
    ]

    for col, title, ax in panels:
        btc_med = get_medians(df_btc, col)
        eth_med = get_medians(df_eth, col)

        ax.plot(hours, btc_med, 'o-', color='#1f77b4', linewidth=2,
                label='BTC', markersize=5, zorder=3)
        ax.plot(hours, eth_med, 's-', color='#ff7f0e', linewidth=2,
                label='ETH', markersize=5, zorder=3)

        # Vertical lines with top-edge text labels instead of legend
        y_top = ax.get_ylim()[1] if ax.get_ylim()[1] != 1.0 else max(
            [v for v in btc_med + eth_med if not np.isnan(v)]) * 1.02
        
        for x, color, ls, label in market_lines:
            ax.axvline(x, color=color, linestyle=ls, linewidth=1.5,
                       alpha=0.85, zorder=2)

        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_xlabel('UTC Hour', fontsize=10)
        ax.set_ylabel(col, fontsize=10)
        ax.set_xticks(range(0, 24, 2))
        ax.grid(True, alpha=0.3, zorder=1)
        ax.legend(fontsize=9, loc='upper left')

    # Single shared legend for market lines — placed below the figure
    from matplotlib.lines import Line2D
    session_handles = [
        Line2D([0], [0], color=c, linestyle=ls, linewidth=1.5, label=lbl)
        for _, c, ls, lbl in market_lines
    ]
    fig.legend(
        handles=session_handles,
        loc='lower center',
        ncol=4,
        fontsize=9,
        frameon=True,
        title='Market sessions (UTC, DST-adjusted)',
        title_fontsize=9,
        bbox_to_anchor=(0.5, -0.02)
    )
    plt.tight_layout(rect=[0, 0.05, 1, 0.97])
    plt.savefig('results/kernel_norms_by_hour.png', dpi=150,
                bbox_inches='tight')
    plt.show()
    print("✓ Saved: results/kernel_norms_by_hour.png")

plot_kernel_norms_by_hour(df_btc, df_eth)

✓ Saved: results/kernel_norms_by_hour.png


/var/folders/x_/4gqtnjds52d1_36fmtw0bv2h0000gn/T/ipykernel_38085/2037993087.py:81: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
